## Step 1. 
#### Calculate the PMI for all successive pairs (w1, w2) of words in the Brown corpus. 

#### Words (not word pairs!) that occur in the corpus less than 10 times should be ignored. 

#### List the 20 word pairs with the highest PMI value and the 20 word pairs with the lowest PMI value. 

In [19]:
# get NGramModel from Part 3

from collections import Counter, defaultdict # You may import more from collections if needed
import regex as re
import numpy as np


class NGramModel:
    def __init__(self, text, n, alpha=0.0):
        """
        Initialize the NGramModel with text and the value of n.
        """
        self.text = text
        self.n = n
        self.alpha = alpha  # Alpha value for additive smoothing
        # Added self.list_ngrams variable to store the ngrams as a list
        self.list_ngrams = []
        self.ngrams = {}
        self.probabilities = {}
        self.vocab = set()
        self.words: list[str] = []

    def tokenize(self) -> list:
        """
        Tokenize the text into words. 
        Fill in the code to split the text into a list of words.
        """
        # Split by whitespace
        tokens = self.text.split()

        return tokens

    def generate_ngrams(self, tokens: list) -> list:
        """
        Generate n-grams from the list of tokens.
        Fill in the code to create n-grams.
        Make sure to treat each sentence independently, include the <s> and </s> tokens.
        """
        # Reset the ngrams list
        self.list_ngrams = []

        # Treat each sentence independently
        sentence = []

        for token in tokens:
            # Start sentence when start of sentence token is encountered
            if token == "<s>":
                sentence = ["<s>"]
                continue
            sentence.append(token)

            if token == "</s>":
                sentence = []
                continue

            if len(sentence) >= self.n:
                # Create the n grams as tuples and add the ngram to the list of ngrams
                self.list_ngrams.append(tuple(sentence[-self.n:]))

        # The template provided for this assignment indicated that this function is meant to return 
        # a list, while self.ngrams is defined as a dictionary. Thus we changed the type hint to dict.
        return self.list_ngrams

    def count_frequencies(self) -> None:
        """
        Count the frequencies of each n-gram.
        Fill in the code to count n-gram occurrences.
        """
        # Use the list of ngrams to populate the dictionary
        for i in self.list_ngrams:
            if i in self.ngrams:
                self.ngrams[i] += 1
            else:
                self.ngrams[i] = 1

    def calculate_probabilities(self) -> None:
        """
        Calculate probabilities of each n-gram based on its frequency. Add alpha smoothing separately.
        """
        # Total number of ngrams in the dataset
        total_ngrams = sum(self.ngrams.values())
        # For smoothing, unique number of n grams
        V = len(self.ngrams)
        for ngram, count in self.ngrams.items():
            self.probabilities[ngram] = (count + self.alpha) / (total_ngrams + (self.alpha * V))

    def most_frequent_ngrams(self, top_n: int = 10) -> list:
        """
        Return the most frequent n-grams and their probabilities.
        """
        # Sort the frequency first
        sorted_ngrams = sorted(self.ngrams.items(), key=lambda x: x[1], reverse=True)[:top_n]
        # Sort the probabilities based on the frequency
        sorted_grams = [(ngram, self.probabilities.get(ngram, 0)) for ngram,_ in sorted_ngrams]
    
        return sorted_grams
    
    def retrieve_words_and_vocab(self) -> None:
        '''Note that the words are lowercased and stripped of punctuation.'''

        words_POS = self.text.split()
        words = [word.rsplit('/', 1)[0] for word in words_POS] # max 2 items; take first item only

        mask = [bool(re.search(r'[a-zA-Z0-9]', word)) for word in words]
        words_array = np.array(words)
        filtered_words = words_array[mask]

        self.words = [re.sub(r'[^a-zA-Z0-9]', '', word).lower() for word in filtered_words]
        self.vocab = set(self.words)

    def return_valid_unigrams(self, n_valid: int = 10) -> set[str]:
        ''' return set of all words appearing at least n_valid times. Only call on unigram model. '''
        if self.n==1:
            valid_words = {ngram[0] for ngram, count in self.ngrams.items() if count >= n_valid}
            return valid_words
        else:
            raise ValueError("return_valid_unigrams can only be called on a unigram model (n=1)")
    
    def return_corpus_size(self) -> int:
        return len(self.words)



In [20]:
import nltk
nltk.download('brown')
brown_corpus = nltk.corpus.brown
brown_text = brown_corpus.raw()

# clean raw text
words_POS = brown_text.split()
clean = [word.rsplit('/', 1)[0] for word in words_POS] # max 2 items; take first item only
removed_punct = [re.sub(r'[^a-zA-Z0-9]', '', word) for word in clean] # treat capitalisation as separate words
brown_text = ' '.join(removed_punct)


# unigram model
unigram_model = NGramModel(brown_text, 1)
tokens = unigram_model.tokenize()
unigram_model.generate_ngrams(tokens)
unigram_model.count_frequencies()
unigram_model.calculate_probabilities()
unigram_model.retrieve_words_and_vocab()

# bigram model for (w1,w2)
bigram_model = NGramModel(brown_text, 2)
bigram_model.generate_ngrams(tokens)  # reuse tokens
bigram_model.count_frequencies()
bigram_model.calculate_probabilities()

# total word count and retrieve valid words
N = unigram_model.return_corpus_size()
valid_words = unigram_model.return_valid_unigrams()

def freq(w) -> int:
    ''' returns absolute frequency of word '''
    return unigram_model.ngrams.get((w,), 0) # returns 0 if not found

def pmi(valid_words, n) -> list[tuple]:
    pmi_list = []
    for w1 in valid_words:
        for w2 in valid_words:
            C12 = bigram_model.ngrams.get((w1, w2), 0)

            if C12 == 0:  # these would cause error with log(0)
                # alternatively, we could sub -infty as a PMI for non-co-occuring words; 
                # though this would be time-consuming
                continue

            C1 = freq(w1)
            C2 = freq(w2)
            pmi = np.log((C12 * n) / (C1 * C2))
            pmi_list.append(((w1, w2), float(pmi)))

    return pmi_list

pmi_list = pmi(valid_words, N)
sorted_pmi = sorted(pmi_list, key=lambda x: -x[1])
print("Top 20 highest PMI:", sorted_pmi[:20])
print("Top 20 lowest PMI:", sorted_pmi[-20:])

''' 
Comment on result: 

We can see that the score approximately ranges form -5.219 to 11.43. 
In theory, the PMI score can range from -infty to infty. 
Word pairs that never appear together are excluded from the PMI calculation, as these word incur division by zero errors.
A possible alternative solution would be to substitute a PMI score of -infty for non-co-occuring pairs.
In our definition of PMI, we excluded -infty scores as these scores seemed meaningless to the authors.
Words that often appear together seem to consist mostly of names like 'Hong Kong' 'Viet Nam' 'Pathet Lao' and more.
'''

[nltk_data] Downloading package brown to /Users/Gileesa/nltk_data...
[nltk_data]   Package brown is already up-to-date!


Top 20 highest PMI: [(('Hong', 'Kong'), 11.430846367079043), (('Viet', 'Nam'), 11.056152917637634), (('Pathet', 'Lao'), 10.995528295821199), (('Simms', 'Purdew'), 10.995528295821199), (('El', 'Paso'), 10.833009366323424), (('7th', 'Cavalry'), 10.833009366323424), (('Herald', 'Tribune'), 10.81180715867282), (('Lo', 'Shu'), 10.784219202153992), (('WTV', 'antigen'), 10.724795781683191), (('Gray', 'Eyes'), 10.699477973698901), (('Internal', 'Revenue'), 10.650687809529469), (('Puerto', 'Rico'), 10.650687809529469), (('decomposition', 'theorem'), 10.496537129702212), (('Saxon', 'Shore'), 10.47883755260281), (('anionic', 'binding'), 10.476334422384692), (('ExportImport', 'Bank'), 10.461445809890941), (('carbon', 'tetrachloride'), 10.442469908431935), (('Common', 'Market'), 10.42754425821526), (('Beverly', 'Hills'), 10.416494422028675), (('Kohnstamm', 'reactivity'), 10.398794844929274)]
Top 20 lowest PMI: [(('as', 'and'), -5.21893574489322), (('in', 'was'), -5.23926525322503), (('his', 'the'),

" \nComment on result: \n\nWe can see that the score approximately ranges form -5.219 to 11.43. \nIn theory, the PMI score can range from -infty to infty. \nWord pairs that never appear together are excluded from the PMI calculation, as these word incur division by zero errors.\nA possible alternative solution would be to substitute a PMI score of -infty for non-co-occuring pairs.\nIn our definition of PMI, we excluded -infty scores as these scores seemed meaningless to the authors.\nWords that often appear together seem to consist mostly of names like 'Hong Kong' 'Viet Nam' 'Pathet Lao' and more.\n"

## Step 2. 
#### Discuss the validity of the independence assumption for unigram models. Give 2-3 examples from your results to support your ideas. (word limit 200)

If two words are truly independent, the probability of find them together can be defined as $P(w1,w2) = P(w1) \cdot P(w2)$ per Bayes' theorem. This would mean all PMI scores should return $\log \frac{(P(w1,w2))}{P(w1) \cdot P(w2)} = \log \frac{(P(w1)P(w2))}{P(w1) \cdot P(w2)} = \log(1) = 0$. From our results, we conclude that this is not the case; some words, such as 'Hong' and 'Kong' have a strong probability of appearing after one another, resulting in a PMI of $11.43$. Contrarily, words like 'his' and 'the' are unlikely to appear after one another and thus have a negative PMI value of $-5.29$. These results show that some words often appear together, while others rarely do.

## Step 3. 
##### Extend step 1 by researching and implementing both PMI and positive pointwise mutual information (PPMI) on the entire Brown corpus and brown100. Document and submit your code with observations as comments in the same file as step 1.

In [23]:
def ppmi(valid_words, n) -> list[tuple]:
    ppmi_list = []
    for w1 in valid_words:
        for w2 in valid_words:
            C12 = bigram_model.ngrams.get((w1, w2), 0)

            if C12 == 0:  # these would cause error with log(0)
                # we first wanted to substitute zero, 
                # but this would mean the program would run for several mintues.

                # ppmi_list.append(((w1, w2), 0))
                continue

            C1 = freq(w1)
            C2 = freq(w2)
            ppmi = max(np.log((C12 * n) / (C1 * C2)), 0) # per Standford source: https://web.stanford.edu/~jurafsky/slp3/J.pdf
            ppmi_list.append(((w1, w2), float(ppmi)))

    return ppmi_list

# apply to entire Brown corpus
ppmi_list = ppmi(valid_words, N)
sorted_ppmi = sorted(ppmi_list, key=lambda x: -x[1])
print("======= FULL BROWN CORPUS =======")
print("Top 20 highest PPMI:", sorted_ppmi[:20])
print("Top 20 lowest PPMI:", sorted_ppmi[-20:])



# apply to brown100 file
with open('./brown_100.txt', 'r') as file_:
    brown100_text = file_.read()

words_POS = brown100_text.split()
clean = [word.rsplit('/', 1)[0] for word in words_POS]
removed_punct = [re.sub(r'[^a-zA-Z0-9]', '', word) for word in clean]
brown100_clean = ' '.join(removed_punct)

unigram_model_100 = NGramModel(brown100_clean, 1)
tokens_100 = unigram_model_100.tokenize()
unigram_model_100.generate_ngrams(tokens_100)
unigram_model_100.count_frequencies()
unigram_model_100.calculate_probabilities()
unigram_model_100.retrieve_words_and_vocab()

bigram_model_100 = NGramModel(brown100_clean, 2)
bigram_model_100.generate_ngrams(tokens_100)
bigram_model_100.count_frequencies()
bigram_model_100.calculate_probabilities()

N_100 = unigram_model_100.return_corpus_size()
valid_words_100 = unigram_model_100.return_valid_unigrams()

ppmi_list_100 = ppmi(valid_words_100, N_100)
sorted_ppmi_100 = sorted(ppmi_list_100, key=lambda x: -x[1])
print("\n======= BROWN100 CORPUS =======")
print("Top 20 highest PPMI:", sorted_ppmi_100[:20])
print("Top 20 lowest PPMI:", sorted_ppmi_100[-20:])

''' 
Comment on result: 

With the PPMI, we observe that the scores from the full Brown Corpus range form 0 to approx. 11.43, 
while those of Brown100 range approx. from  0.85 to 4.5. 
In theory, the PPMI score can range from 0 to infty. 

The PPMI caps negative scores to 0 because these are often unreliable, except for extremely large corpora.
Furthermore, negative scores indicate 'unrelatedness', which is hard for human to interpret.

The Brown100 corpus is of a much smaller size, meaning that the lowest theoretical score of 0 does not appear at all. 
For the same reason, the highest found score is lower than that found in the full Brown corpus. This is becaue 
a smaller corpus means that rare word pairs are found together less often, so the highest PMI pairs are less extreme.

Word pairs that never appear together are still not included, as substituting 0 for each non-co-occuring pair would make
the program extremely time-consuming to run.
'''

======= FULL BROWN CORPUS =======
Top 20 highest PPMI: [(('Hong', 'Kong'), 11.430846367079043), (('Viet', 'Nam'), 11.056152917637634), (('Pathet', 'Lao'), 10.995528295821199), (('Simms', 'Purdew'), 10.995528295821199), (('El', 'Paso'), 10.833009366323424), (('7th', 'Cavalry'), 10.833009366323424), (('Herald', 'Tribune'), 10.81180715867282), (('Lo', 'Shu'), 10.784219202153992), (('WTV', 'antigen'), 10.724795781683191), (('Gray', 'Eyes'), 10.699477973698901), (('Internal', 'Revenue'), 10.650687809529469), (('Puerto', 'Rico'), 10.650687809529469), (('decomposition', 'theorem'), 10.496537129702212), (('Saxon', 'Shore'), 10.47883755260281), (('anionic', 'binding'), 10.476334422384692), (('ExportImport', 'Bank'), 10.461445809890941), (('carbon', 'tetrachloride'), 10.442469908431935), (('Common', 'Market'), 10.42754425821526), (('Beverly', 'Hills'), 10.416494422028675), (('Kohnstamm', 'reactivity'), 10.398794844929274)]
Top 20 lowest PPMI: [(('cook', 'the'), 0.0), (('broke', 'to'), 0.0), (('b

" \nComment on result: \n\nWith the PPMI, we observe that the scores from the full Brown Corpus range form 0 to approx. 11.43, \nwhile those of Brown100 range approx. from  0.85 to 4.5. \nIn theory, the PPMI score can range from 0 to infty. \n\nThe PPMI caps negative scores to 0 because these are often unreliable, except for extremely large corpora.\nFurthermore, negative scores indicate 'unrelatedness', which is hard for human to interpret.\n\nThe Brown100 corpus is of a much smaller size, meaning that the lowest theoretical score of 0 does not appear at all. \nFor the same reason, the highest found score is lower than that found in the full Brown corpus. This is becaue \na smaller corpus means that rare word pairs are found together less often, so the highest PMI pairs are less extreme.\n\nWord pairs that never appear together are still not included, as substituting 0 for each non-co-occuring pair would make\nthe program extremely time-consuming to run.\n"